In [1]:
import matplotlib
%matplotlib inline
from eval_utils_multilingualv2 import get_text_processors, get_s2t, get_prompt, get_item, sample
from utils import set_hparams, HParams, load_checkpoint, load_a2flow, load_dp, load_vocoder

In [24]:
import ndjson

manifest = [{
    # path to the sample to clone
    'audio_filepath': '/lustre/fsw/portfolios/edgeai/users/amhussein/tts_test_output_dir/prompts/_es_57baaa4728117235bc384c6a9094b3eaf3c24c8f_dd1e3172babfd326b127e583b6f4f96c_voxpopuli_20190314-0900-PLENARY-es_20190314-13:40:27_2_000004.wav',
    # text of the sample to clone
    'transcript': 'Y asegurar las libertades sociales y políticas más básicas.',
    # [optional] duration in seconds of the sample to clone (must be ~5 seconds)
    'duration': '3.349',
    # text to synthesize
    'text': 'And to ensure the most basic social and political freedoms.',
    # language of the text to synthesize
    'language': 'en_US',
    # language of the prompt text + audio
    'language_in': 'es_MX'
},]



# Save to file
with open("sample_manifest.ndjson", "w", encoding="utf-8") as f:
    ndjson.dump(manifest, f)

In [25]:
base_path = "/lustre/fsw/portfolios/edgeai/users/amhussein"
args = HParams()
args.generator_path = f"{base_path}/pretrained_models/a2flow/a2f-multilingual-v2"
# ckpt_num = 170000
ckpt_num = 275000

hps_generator = set_hparams(args.generator_path)

# dp for English
args.dp_path = f"{base_path}/pretrained_models/a2flow/a2flow-dp-en"
hps_dp = set_hparams(args.dp_path)

vocoder_path = f"{base_path}/pretrained_models/a2flow/bigvgan_v2.pt"
vocoder_config_path = f"{base_path}/pretrained_models/a2flow/bigvgan_v2-config.json"

In [26]:
import ndjson
# manifest_path = "/scripts/multilingual_mini.ndjson"
manifest_path = "sample_manifest.ndjson"
with open(manifest_path, 'r', encoding="utf-8") as fin:
    data = ndjson.load(fin)
# data_filtered = [d for d in data if d['language'] in ['es_ES', 'de_DE']]
data_filtered = data

In [27]:
# sample manifest entry
{
    # path to the sample to clone
    'audio_filepath': 'multilingual/ref_sample.wav',
    # text of the sample to clone
    'transcript': 'Bless you! Here, take these. The city air can be a bit overwhelming at first.',
    # [optional] duration in seconds of the sample to clone (must be ~5 seconds)
    'duration': '05.52',
    # text to synthesize
    'text': 'Meine Mutter ist ein Zwilling und ich finde es super cool!',
    # language of the text to synthesize
    'language': 'de_DE',
    # language of the prompt text + audio
    'language_in': 'en_US'
}

# longer estimate audio duration of prompt + synth is more likely to hallucinate
# for this particular checkpoint, ~20s total should be okay

{'audio_filepath': 'multilingual/ref_sample.wav',
 'transcript': 'Bless you! Here, take these. The city air can be a bit overwhelming at first.',
 'duration': '05.52',
 'text': 'Meine Mutter ist ein Zwilling und ich finde es super cool!',
 'language': 'de_DE',
 'language_in': 'en_US'}

In [28]:
data_filtered

[{'audio_filepath': '/lustre/fsw/portfolios/edgeai/users/amhussein/tts_test_output_dir/prompts/_es_57baaa4728117235bc384c6a9094b3eaf3c24c8f_dd1e3172babfd326b127e583b6f4f96c_voxpopuli_20190314-0900-PLENARY-es_20190314-13:40:27_2_000004.wav',
  'transcript': 'Y asegurar las libertades sociales y políticas más básicas.',
  'duration': '3.349',
  'text': 'And to ensure the most basic social and political freedoms.',
  'language': 'en_US',
  'language_in': 'es_MX'}]

In [29]:
text_processors = get_text_processors("configs/text_processors_config_prondict.json")

In [30]:
generator = load_a2flow(args.generator_path, hps_generator, n_language=len(text_processors), num=ckpt_num).cuda().eval()        
dp = load_dp(args.dp_path, hps_dp, n_language=len(text_processors)).cuda().eval()
vocoder = load_vocoder(vocoder_config_path, vocoder_path).cuda()

2025-07-09 20:27:14 | INFO | fairseq.tasks.hubert_pretraining | current directory is /lustre/fs12/portfolios/edgeai/projects/edgeai_riva_rivamlops/users/amhussein/toolkits/a-2-flow-inference-fork
2025-07-09 20:27:14 | INFO | fairseq.tasks.hubert_pretraining | HubertPretrainingTask Config {'_name': 'hubert_pretraining', 'data': '/checkpoint/wnhsu/data/CM3/multilingual/speech/', 'fine_tuning': False, 'labels': ['km'], 'label_dir': '/checkpoint/wnhsu/experiments/hubert/kmeans/mhubert_vp_mls_cv_8lang_it2/cm3/vp_mls_cv_8lang.layer9.km1000', 'label_rate': 50.0, 'sample_rate': 16000, 'normalize': False, 'enable_padding': False, 'max_keep_size': None, 'max_sample_size': 250000, 'min_sample_size': 32000, 'single_target': False, 'random_crop': True, 'pad_audio': False}
2025-07-09 20:27:14 | INFO | fairseq.models.hubert.hubert | HubertModel Config: {'_name': 'hubert', 'label_rate': 50.0, 'extractor_mode': default, 'encoder_layers': 12, 'encoder_embed_dim': 768, 'encoder_ffn_embed_dim': 3072, 'enc

Removing weight norm of BigVGAN...


In [31]:
import wave
import contextlib

def get_wav_duration(fname):
    with contextlib.closing(wave.open(fname,'r')) as f:
        frames = f.getnframes()
        rate = f.getframerate()
        return frames / float(rate)

In [32]:
args.timesteps = 32
args.gradient_scale = 2.
args.alpha = 3.
args.fp16 = False
args.dur_scale = 1. 
args.display_audio = True

use_whisper = False
# wav_path_base = "/lustre/fsw/portfolios/edgeai/users/marianag/a2f_eval/"

for i, d in enumerate(data_filtered):
    args.prompt_lang = d["language_in"]
    args.synth_lang = d["language"]
    
    if not "duration" in d.keys():
        wav_duration = get_wav_duration(f"{wav_path_base}/{d['audio_filepath']}")
        d["duration"] = wav_duration
    
    # Load Prompt
    # wav, mel, transcript = get_prompt(
    #     sample_path=f"{wav_path_base}/{d['audio_filepath']}", 
    #     num_second=float(d["duration"]),
    #     transcript=d["transcript"], 
    #     use_whisper=use_whisper
    # )
    wav, mel, transcript = get_prompt(
        sample_path=d['audio_filepath'], 
        num_second=float(d["duration"]),
        transcript=d["transcript"], 
        use_whisper=use_whisper
    )

    text = d["text"]
    audio = sample(
        args, get_item([text], wav, mel, transcript), 
       text_processors, generator, dp, vocoder
    )
    

Load HuBERT-L to transcribe the prompt
Transcript of Prompt: Y asegurar las libertades sociales y políticas más básicas.
Prompt Audio


duration_scale: 1.0
gen_len tensor([259.2258], device='cuda:0')
Text input: And to ensure the most basic social and political freedoms.
Generated Audio


In [9]:
args.generator_path

'/workspace/pretrained_models/a2flow/a2f-multilingual-not-long'

In [10]:
!ls /workspace/pretrained_models/a2flow/a2f-multilingual-v2

config.json  fm_110000.pt
